In [18]:
import pandas as pd
import os

folder = os.getcwd()
dfs = []

month_map = {
    "Jan": "January",
    "Feb": "February",
    "Mar": "March",
    "Apr": "April",
    "May": "May",
    "Jun": "June",
    "Jul": "July",
    "Aug": "August",
    "Sep": "September",
    "Oct": "October",
    "Nov": "November",
    "Dec": "December"
}

month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

for file in os.listdir(folder):
    if file.endswith(".csv") and "2018" in file and "reformatted" not in file:
        try:
            month_abbr = file.split(",")[1].strip().split(" ")[0]
            month = month_map[month_abbr]
        except Exception:
            print(f"Skipping unexpected filename: {file}")
            continue

        df = pd.read_csv(os.path.join(folder, file))

        needed_cols = ["County", "County Code", "Deaths"]
        if not all(col in df.columns for col in needed_cols):
            print(f"Skipping file with wrong columns: {file}")
            print("Columns found:", df.columns.tolist())
            continue

        df = df[["County", "County Code", "Deaths"]].copy()

        df["County"] = df["County"].astype("string").str.strip()
        df["County Code"] = df["County Code"].astype("string").str.strip()
        df["Deaths"] = df["Deaths"].astype("string").str.strip()

        df["County"] = df["County"].replace({
            "": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA
        })
        df["County Code"] = df["County Code"].replace({
            "": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA
        })
        df["Deaths"] = df["Deaths"].replace({
            "": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA
        })

        df["Deaths"] = df["Deaths"].replace({
            "suppressed": "Suppressed",
            "SUPPRESSED": "Suppressed"
        })

        df["Month"] = month
        dfs.append(df)

if not dfs:
    raise ValueError("No matching 2018 CSV files were found in this folder.")

combined = pd.concat(dfs, ignore_index=True)

combined["County"] = combined["County"].replace({
    "": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA
})
combined["County Code"] = combined["County Code"].replace({
    "": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA
})

combined = combined.dropna(subset=["County", "County Code"])
combined = combined[combined["County"].str.strip() != ""]
combined = combined[combined["County Code"].str.strip() != ""]

combined = combined[~combined["County"].str.contains(
    "Total|All Counties|United States",
    case=False,
    na=False
)]

combined = combined.drop_duplicates(subset=["County", "Month"], keep="first")

county_lookup = (
    combined[["County", "County Code"]]
    .drop_duplicates()
    .sort_values("County")
)

all_counties = county_lookup["County"].tolist()

full_grid = pd.MultiIndex.from_product(
    [all_counties, month_order],
    names=["County", "Month"]
).to_frame(index=False)

full_df = full_grid.merge(
    combined[["County", "Month", "Deaths"]],
    on=["County", "Month"],
    how="left"
)

full_df = full_df.merge(county_lookup, on="County", how="left")

result = full_df.pivot(
    index=["County", "County Code"],
    columns="Month",
    values="Deaths"
).reset_index()

result = result[["County", "County Code"] + month_order]

result[month_order] = result[month_order].fillna(0)

result = result[result["County"].notna()]
result = result[result["County"].astype("string").str.strip() != ""]
result = result[result["County"].astype("string").str.strip().str.lower() != "nan"]

for col in month_order:
    result[col] = result[col].apply(
        lambda x: str(int(float(x))) if pd.notna(x) and str(x).replace(".", "", 1).isdigit() else x
    )

# Save
result.to_csv("2018 Lung Cancer Death Rates_reformatted.csv", index=False)